# Atelier PyTorch Mistral · exercices des 12 séances

Ce notebook suit exactement les séances du parcours : une section par séance, dans le même ordre.
1. Exécute la cellule du harnais (une fois par session Colab).
2. Pour chaque exercice : `start("id")` lance le chrono, complète la cellule, puis `check("id")` lance les tests.
3. `hint("id")` donne un indice à la fois ; `solution("id")` affiche la solution commentée.

**Conditions d'entretien** : désactive les suggestions d'IA de Colab (Paramètres, section IA) et annonce les shapes à voix haute.

In [ ]:
#@title ⚙️ Harnais de correction (exécute cette cellule en premier)
import base64, zlib
_B = (
    "..."
)
exec(zlib.decompress(base64.b64decode(_B)).decode("utf-8"))

# Séance 1 · Tenseurs et PyTorch

Manipuler les shapes sans hésiter : view et reshape, broadcasting, einsum, gather, masques. C'est le socle de tous les exercices de code.

## Produit extérieur par broadcasting  `t01`

⏱ objectif 4 min

outer(a, b) : (n,) et (m,) → (n, m), sans torch.outer. Sans boucle Python.

In [ ]:
def outer(a, b):
    """a : (n,), b : (m,) → (n, m) avec out[i, j] = a[i] * b[j]. Sans boucle, sans torch.outer."""
    raise NotImplementedError

In [ ]:
check("t01")

## Masque causal sans tril  `t02`

⏱ objectif 4 min

causal_mask(T) → booléen (T, T), True si j ≤ i. Sans boucle Python.

In [ ]:
def causal_mask(T):
    """Booléen (T, T) : True en (i, j) si le token i a le droit de voir j (j ≤ i). Sans torch.tril."""
    raise NotImplementedError

In [ ]:
check("t02")

## Masque de sliding window  `t03`

⏱ objectif 4 min

window_mask(T, W) → True si 0 ≤ i − j < W (Mistral 7B : W = 4096). Sans boucle Python.

In [ ]:
def window_mask(T, W):
    """Masque de la sliding window (Mistral 7B) : True si 0 ≤ i − j < W."""
    raise NotImplementedError

In [ ]:
check("t03")

## Softmax stable  `t04`

⏱ objectif 4 min

Softmax sur la dernière dimension, correcte même pour des logits de l'ordre de 1000. Sans boucle Python.

In [ ]:
def softmax_stable(x):
    """Softmax sur la dernière dimension, stable pour des valeurs énormes. Sans torch.softmax."""
    raise NotImplementedError

In [ ]:
check("t04")

## logsumexp stable  `t05`

⏱ objectif 4 min

log Σ exp(x) sur la dernière dimension, sans torch.logsumexp. Sans boucle Python.

In [ ]:
def logsumexp_stable(x):
    """log Σ exp(x) sur la dernière dimension, stable. Sans torch.logsumexp."""
    raise NotImplementedError

In [ ]:
check("t05")

## One-hot sans F.one_hot  `t06`

⏱ objectif 4 min

one_hot(idx, n) : (B,) → (B, n) float. Sans boucle Python.

In [ ]:
def one_hot(idx, n):
    """idx : (B,) entiers → (B, n) float avec un 1 par ligne. Sans F.one_hot."""
    raise NotImplementedError

In [ ]:
check("t06")

## Lire un logit par ligne  `t07`

⏱ objectif 4 min

pick(logits, targets)[b] = logits[b, targets[b]] (cœur de la cross-entropy). Sans boucle Python.

In [ ]:
def pick(logits, targets):
    """logits : (B, V), targets : (B,) → (B,) avec out[b] = logits[b, targets[b]]."""
    raise NotImplementedError

In [ ]:
check("t07")

## Scores d'attention avec einsum  `t09`

⏱ objectif 4 min

q (B, H, T, d) et k (B, H, S, d) → (B, H, T, S) avec torch.einsum. Sans boucle Python.

In [ ]:
def attn_scores(q, k):
    """q : (B, H, T, d), k : (B, H, S, d) → scores (B, H, T, S) avec torch.einsum."""
    raise NotImplementedError

In [ ]:
check("t09")

## Masque top-k  `t11`

⏱ objectif 4 min

Garde les k plus grands logits par ligne, −inf ailleurs. Sans boucle Python.

In [ ]:
def topk_mask(logits, k):
    """Garde les k plus grands logits de chaque ligne, met −inf ailleurs."""
    raise NotImplementedError

In [ ]:
check("t11")

## Découper et recoller les têtes  `t12`

⏱ objectif 4 min

split (B, T, D) → (B, H, T, D/H) et merge, l'inverse. Sans boucle Python.

In [ ]:
def split_heads(x, H):
    """(B, T, D) → (B, H, T, D // H)"""
    raise NotImplementedError

def merge_heads(x):
    """(B, H, T, dh) → (B, T, H·dh)"""
    raise NotImplementedError

In [ ]:
check("t12")

## Dernier vrai token  `t13`

⏱ objectif 4 min

x (B, T, D) et lengths (B,) → (B, D), le dernier token non paddé. Sans boucle Python.

In [ ]:
def last_token(x, lengths):
    """x : (B, T, D), lengths : (B,) longueurs réelles (padding à droite) → (B, D) : le dernier vrai token."""
    raise NotImplementedError

In [ ]:
check("t13")

## Positions avec padding à gauche  `t14`

⏱ objectif 4 min

mask (B, T) → positions 0, 1, 2… à partir du premier vrai token. Sans boucle Python.

In [ ]:
def positions_from_mask(mask):
    """mask : (B, T) avec 0 = padding (à gauche), 1 = vrai token → positions (B, T) :
    0, 1, 2… à partir du premier vrai token, 0 sur le padding."""
    raise NotImplementedError

In [ ]:
check("t14")

# Séance 2 · Softmax, cross-entropy, KL

Écrire log-softmax, cross-entropy et KL de façon stable, et savoir les relier.

## log-softmax, cross-entropy et KL  `ce_kl`

⏱ objectif 15 min

Écris log_softmax (stable), cross_entropy(logits (N, V), targets (N,), ignore_index=-100) qui moyenne sur les seules cibles gardées, et kl_div(p_logits, q_logits) = KL(p‖q) moyenné sur les lignes. Un candidat stagiaire a eu « cross-entropy vs KL » dès le premier appel.

In [ ]:
start("ce_kl")

In [ ]:
def log_softmax(x, dim=-1):
    """log softmax stable (sans F.log_softmax)."""
    raise NotImplementedError

def cross_entropy(logits, targets, ignore_index=-100):
    """logits : (N, V), targets : (N,) → perte moyenne sur les cibles ≠ ignore_index."""
    raise NotImplementedError

def kl_div(p_logits, q_logits):
    """KL(p ‖ q) à partir de logits (B, V), moyenne sur les lignes."""
    raise NotImplementedError

In [ ]:
check("ce_kl")

# Séance 3 · Attention causale multi-têtes

Écrire de mémoire, en moins de 20 minutes, une attention multi-têtes causale et batchée, la tester et donner sa complexité.

## Attention causale multi-têtes  `mha`

⏱ objectif 20 min

CausalSelfAttention(d_model, n_heads) avec wq, wk, wv, wo (nn.Linear sans biais). forward(x : (B, T, d_model)) → (B, T, d_model). Batché, masque causal, sans F.scaled_dot_product_attention. C'est la question exacte rapportée par un candidat Mistral : « Implement Multi-Headed Self-Attention from scratch (batched) with a Causal Mask ».

In [ ]:
start("mha")

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # TODO : self.wq, self.wk, self.wv, self.wo = nn.Linear(..., bias=False)
        raise NotImplementedError

    def forward(self, x):
        """x : (B, T, d_model) → (B, T, d_model). Masque causal, sans F.scaled_dot_product_attention."""
        raise NotImplementedError

In [ ]:
check("mha")

## Attention multi-têtes  `d2`

⏱ objectif 10 min

Cinq bugs classiques dans une attention : trouve-les avant de lancer le test.

**5 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d2")

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d, n_heads, max_len):
        super().__init__()
        self.h, self.dh = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.register_buffer("mask", torch.tril(torch.ones(max_len, max_len)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.view(B, T, self.h, self.dh).transpose(1, 2)
        k = k.view(B, T, self.h, self.dh).transpose(1, 2)
        v = v.view(B, T, self.h, self.dh)
        att = q @ k.transpose(-2, -1) / self.dh
        att = att.masked_fill(self.mask[:T, :T] == 0, 0.0)
        att = att.softmax(dim=-2)
        y = (att @ v).transpose(1, 2).view(B, T, C)
        return self.proj(y)

In [ ]:
check("d2")

# Séance 4 · GQA et mémoire du KV cache

Passer de la MHA à la GQA, et savoir calculer la mémoire du KV cache d'un modèle comme Mistral 7B.

## Grouped-query attention  `gqa`

⏱ objectif 15 min

GQAttention(d_model, n_heads, n_kv_heads) : wq vers n_heads·dh, wk et wv vers n_kv_heads·dh. La tête de query h utilise la tête K/V h // (n_heads / n_kv_heads). Doit aussi marcher en MHA (n_kv = n_heads) et en MQA (n_kv = 1).

In [ ]:
start("gqa")

In [ ]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        # TODO : wq (d_model → n_heads·dh), wk et wv (d_model → n_kv_heads·dh), wo
        raise NotImplementedError

    def forward(self, x):
        """Attention causale : chaque groupe de n_heads // n_kv_heads têtes de query partage une tête K/V."""
        raise NotImplementedError

In [ ]:
check("gqa")

## repeat_kv pour la GQA  `t10`

⏱ objectif 4 min

(B, H_kv, T, d) → (B, H_kv·n_rep, T, d) avec expand + reshape. Sans boucle Python.

In [ ]:
def repeat_kv(x, n_rep):
    """x : (B, H_kv, T, d) → (B, H_kv·n_rep, T, d) ; la tête h de la sortie est la tête h // n_rep de x.
    Utilise expand + reshape (sans repeat_interleave)."""
    raise NotImplementedError

In [ ]:
check("t10")

## Refaire de zéro : Attention causale multi-têtes  `mha`

Sans regarder ta version précédente : ajoute une cellule, recopie le squelette depuis la séance 3, `start("mha")`, puis `check("mha")`.

# Séance 5 · Normalisation et RoPE

Écrire LayerNorm, RMSNorm et RoPE, et expliquer pre-norm et l'encodage relatif des positions.

## LayerNorm et RMSNorm  `norms`

⏱ objectif 10 min

LayerNorm(dim, eps=1e-5) avec weight (uns) et bias (zéros) ; RMSNorm(dim, eps=1e-6) avec weight seulement, calcul en float32 et sortie au dtype d'entrée, comme dans mistral-inference.

In [ ]:
start("norms")

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        # TODO : self.weight (uns), self.bias (zéros), self.eps
        raise NotImplementedError

    def forward(self, x):
        # normalise sur la dernière dimension : (x − μ) / √(σ² + ε) · w + b
        raise NotImplementedError


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        # TODO : self.weight (uns), self.eps — pas de biais
        raise NotImplementedError

    def forward(self, x):
        # x / √(mean(x²) + ε) · w, calcul en float32, sortie au dtype de x
        raise NotImplementedError

In [ ]:
check("norms")

## Rotary position embeddings  `rope`

⏱ objectif 20 min

precompute_freqs(dh, max_len, theta=10000) → (cos, sin) de shape (max_len, dh/2) avec θ_i = theta^(−2i/dh). apply_rope(x (B, H, T, dh), cos, sin, start_pos=0) fait tourner chaque paire entrelacée (x[2i], x[2i+1]) de l'angle (start_pos + t)·θ_i, en float32, sortie au dtype de x. C'est la convention de mistral-inference (view_as_complex).

In [ ]:
start("rope")

In [ ]:
def precompute_freqs(dh, max_len, theta=10000.0):
    """Renvoie cos, sin de shape (max_len, dh // 2) pour les angles m·θ_i, θ_i = theta^(−2i/dh)."""
    raise NotImplementedError

def apply_rope(x, cos, sin, start_pos=0):
    """x : (B, H, T, dh). Fait tourner chaque paire (x[2i], x[2i+1]) de l'angle de la position
    start_pos + t. Calcul en float32, sortie au dtype de x."""
    raise NotImplementedError

In [ ]:
check("rope")

## RMSNorm et RoPE  `d6`

⏱ objectif 8 min

Une RMSNorm et une RoPE réécrites de mémoire, quatre erreurs.

**4 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d6")

In [ ]:
class RMSNorm2(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.zeros(d))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True) + self.eps
        return x / rms * self.weight

def rope_tables(dh, T, theta=10000.0):
    freqs = 1.0 / theta ** (torch.arange(dh // 2) / dh)
    ang = torch.arange(T)[:, None] * freqs[None, :]
    return ang.cos(), ang.sin()

def rope_apply(x, cos, sin):            # x : (B, H, T, dh), paires entrelacées
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos + x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

In [ ]:
check("d6")

# Séance 6 · Le bloc Transformer et le modèle complet

Assembler SwiGLU, attention et normes en un bloc pre-norm, puis en modèle complet, et compter ses paramètres.

## FFN SwiGLU  `swiglu`

⏱ objectif 6 min

FeedForward(d_model, d_ff) : w2(silu(w1 x) ⊙ w3 x), trois nn.Linear sans biais (noms de mistral-inference).

In [ ]:
start("swiglu")

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO : w1 (gate), w3 (up) : d_model → d_ff ; w2 (down) : d_ff → d_model ; sans biais
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

In [ ]:
check("swiglu")

## Bloc Transformer et modèle complet  `block`

⏱ objectif 30 min

Attention(d_model, n_heads, n_kv_heads) avec forward(x, cos, sin) (GQA + RoPE + causal). TransformerBlock(d_model, n_heads, n_kv_heads, d_ff) avec attention, feed_forward, attention_norm, ffn_norm, en pre-norm. Transformer(vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_len, theta, tie_weights=True) avec tok_embeddings, layers, norm, output. Le pair programming rapporté portait justement sur un bug de pre-norm et de résiduels dans un bloc.

In [ ]:
start("block")

In [ ]:
# Réutilise RMSNorm, precompute_freqs, apply_rope, FeedForward des drills précédents,
# ou réécris tout (conditions réelles d'entretien).

class Attention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        raise NotImplementedError   # wq, wk, wv, wo, n_heads, n_kv_heads

    def forward(self, x, cos, sin):
        """GQA + RoPE (paires entrelacées) + masque causal."""
        raise NotImplementedError


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff):
        super().__init__()
        raise NotImplementedError   # attention, feed_forward, attention_norm, ffn_norm

    def forward(self, x, cos, sin):
        raise NotImplementedError   # pre-norm + connexions résiduelles


class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_len,
                 theta=10000.0, tie_weights=True):
        super().__init__()
        raise NotImplementedError   # tok_embeddings, layers, norm, output, tables RoPE en buffers

    def forward(self, idx):
        """idx : (B, T) entiers → logits (B, T, vocab_size)"""
        raise NotImplementedError

In [ ]:
check("block")

## Bloc Transformer pre-norm  `d1`

⏱ objectif 10 min

Ce bloc pre-norm a 4 bugs : masque, flux résiduel, sortie de nn.MultiheadAttention, device. C'est le type de bug rapporté au pair programming.

**4 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d1")

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        T = x.size(1)
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
        x = self.norm1(x)
        x = x + self.attn(x, x, x, attn_mask=mask)
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
check("d1")

## Refaire de zéro : Rotary position embeddings  `rope`

Sans regarder ta version précédente : ajoute une cellule, recopie le squelette depuis la séance 5, `start("rope")`, puis `check("rope")`.

# Séance 7 · Génération : sampling, KV cache, serving

Écrire l'échantillonnage et une boucle de génération avec KV cache, et comprendre ce qui limite l'inférence.

## Température, top-k, top-p  `sampling`

⏱ objectif 15 min

filter_logits(logits (B, V), top_k, top_p) applique top-k puis top-p (sur la distribution après top-k) en mettant −inf aux tokens retirés ; top-p garde un token si la masse des tokens plus probables que lui est < p. sample_next(logits, temperature, top_k, top_p, generator) → (B,), argmax si temperature = 0.

In [ ]:
start("sampling")

In [ ]:
def filter_logits(logits, top_k=None, top_p=None):
    """logits : (B, V). Applique top-k puis top-p (sur la distribution après top-k).
    Top-p : garde le plus petit ensemble de tokens dont la masse cumulée atteint p
    (un token est gardé si la masse des tokens plus probables que lui est < p).
    Les tokens retirés valent −inf."""
    raise NotImplementedError

def sample_next(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    """logits : (B, V) → (B,) ids. temperature = 0 → argmax. Ordre : température → top-k → top-p."""
    raise NotImplementedError

In [ ]:
check("sampling")

## KV cache et boucle de génération  `kv_cache`

⏱ objectif 30 min

CacheAttention(d_model, n_heads, n_kv_heads, max_batch, max_len) avec buffers cache_k, cache_v (max_batch, n_kv_heads, max_len, dh). forward(x, cos, sin, start_pos) : RoPE aux positions absolues, écriture dans le cache, attention sur toutes les keys ≤ position de chaque query (marche aussi pour un prefill par blocs). generate(model, idx, max_new_tokens, temperature, top_k, top_p) : prefill en un appel model(idx, 0), puis un seul token par appel model(nxt, pos). Tu peux utiliser ref.apply_rope si tu n'as pas fait le drill RoPE.

In [ ]:
start("kv_cache")

In [ ]:
class CacheAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_batch, max_len):
        super().__init__()
        # TODO : wq, wk, wv, wo (GQA) et deux buffers cache_k, cache_v de shape
        #        (max_batch, n_kv_heads, max_len, dh) initialisés à zéro
        raise NotImplementedError

    def forward(self, x, cos, sin, start_pos):
        """x : (B, T, d_model) = les tokens aux positions start_pos … start_pos+T−1.
        cos, sin : tables complètes (max_len, dh/2). Écrit k, v dans le cache, puis
        fait attendre chaque query sur toutes les keys ≤ sa position absolue."""
        raise NotImplementedError


@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    """model(tokens, start_pos) → logits (B, T, V), avec un KV cache interne.
    Prefill du prompt en un appel, puis un seul token par appel."""
    raise NotImplementedError

In [ ]:
check("kv_cache")

## Génération avec KV cache  `d4`

⏱ objectif 10 min

La génération utilise un modèle avec cache interne. Quatre bugs, dont deux silencieux.

**4 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d4")

In [ ]:
@torch.no_grad()
def generate_tokens(model, idx, n_new, temperature=1.0, top_k=None):
    """model(tokens, start_pos) → logits (B, T, V) avec un KV cache interne."""
    logits = model(idx, start_pos=0)
    pos = 0
    for _ in range(n_new):
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1) / temperature
        if top_k is not None:
            v, _ = torch.topk(probs, top_k)
            probs[probs < v[:, [0]]] = 0
        nxt = torch.multinomial(probs, 1)
        idx = torch.cat([idx, nxt], dim=1)
        logits = model(idx, start_pos=pos)
        pos += 1
    return idx

In [ ]:
check("d4")

# Séance 8 · La boucle d'entraînement

Écrire une boucle d'entraînement complète et propre, la déboguer, et répondre aux questions de précision, de mémoire et de parallélisme.

## Boucle d'entraînement complète  `train`

⏱ objectif 30 min

get_lr (warmup linéaire puis cosinus), configure_optimizer (AdamW betas (0.9, 0.95), eps 1e-8, pas de weight decay pour dim < 2), clip_grad_norm (norme globale, renvoie la norme avant clipping), train(model, batches, total_steps, accum_steps, lr_max, lr_min, warmup, weight_decay, clip, device, use_amp) qui renvoie les pertes moyennes par pas, et evaluate(model, batches, n_batches). Le test compare tes poids à une référence après 3 pas.

In [ ]:
start("train")

In [ ]:
def get_lr(step, warmup, total, lr_max, lr_min):
    """Warmup linéaire : lr_max·(step+1)/warmup pour step < warmup ; puis cosinus de lr_max à lr_min
    sur [warmup, total) ; lr_min pour step ≥ total."""
    raise NotImplementedError

def configure_optimizer(model, lr, weight_decay):
    """AdamW(betas=(0.9, 0.95), eps=1e-8), deux groupes : weight decay sur les paramètres de dim ≥ 2,
    0 sur les biais et poids de normes (dim < 2). Paramètres gelés exclus."""
    raise NotImplementedError

def clip_grad_norm(params, max_norm):
    """Clipping par norme globale (comme torch.nn.utils.clip_grad_norm_) ; renvoie la norme avant clipping."""
    raise NotImplementedError

def train(model, batches, total_steps, accum_steps, lr_max, lr_min, warmup,
          weight_decay=0.1, clip=1.0, device="cpu", use_amp=False):
    """batches : itérateur de (x, y) ; model(x) → logits (B, T, V). À chaque pas d'optimisation :
    fixer le lr, accumuler accum_steps micro-batches (perte / accum_steps), clipper, step.
    Renvoie la liste des pertes moyennes par pas (floats)."""
    raise NotImplementedError

def evaluate(model, batches, n_batches):
    """Perte moyenne sur n_batches, en eval() et sans gradient ; remet le mode d'origine."""
    raise NotImplementedError

In [ ]:
check("train")

## Boucle d'entraînement  `d3`

⏱ objectif 12 min

La boucle tourne sans erreur et la perte baisse… mais le modèle n'apprend pas la bonne chose. Huit bugs.

**8 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d3")

In [ ]:
def train_loop(model, batches, steps, lr=3e-3):
    """batches : itérateur de séquences (B, T+1). Le modèle doit prédire le token suivant."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    history = []
    for step in range(steps):
        batch = next(batches)
        x, y = batch[:, :-1], batch[:, :-1]
        logits = model(x)
        V = logits.size(-1)
        loss = F.cross_entropy(logits.softmax(-1).view(-1, V), y.reshape(-1))
        loss.backward()
        opt.step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        history.append(loss)
    return history

def evaluate_loop(model, batches, n_batches):
    total = 0.0
    for _ in range(n_batches):
        batch = next(batches)
        x, y = batch[:, :-1], batch[:, 1:]
        logits = model(x)
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1)).item()
    return total / n_batches

In [ ]:
check("d3")

## Refaire de zéro : Bloc Transformer et modèle complet  `block`

Sans regarder ta version précédente : ajoute une cellule, recopie le squelette depuis la séance 6, `start("block")`, puis `check("block")`.

# Séance 9 · K-Means vectorisé et ML classique

Implémenter K-Means sans boucle sur les points, le déboguer, et faire un BFS rapide.

## K-Means vectorisé  `kmeans`

⏱ objectif 15 min

kmeans_assign(X (N, d), C (K, d)) → (N,) sans tenseur (N, K, d). kmeans(X, K, n_iter, seed) → (C, assign) cohérents, cluster vide géré. Question rapportée telle quelle par un candidat stagiaire : « how do you efficiently implement the prediction function for K-Means ? ».

In [ ]:
start("kmeans")

In [ ]:
def kmeans_assign(X, C):
    """X : (N, d), C : (K, d) → (N,) indice du centroïde le plus proche. Vectorisé, mémoire O(N·K)."""
    raise NotImplementedError

def kmeans(X, K, n_iter=20, seed=0):
    """Algorithme de Lloyd. Renvoie (C (K, d), assign (N,)) cohérents : assign = kmeans_assign(X, C).
    Un cluster vide garde son ancien centroïde."""
    raise NotImplementedError

In [ ]:
check("kmeans")

## K-Means  `d5`

⏱ objectif 8 min

Le K-Means vectorisé demandé en entretien de stage, avec quatre bugs.

**4 bugs à trouver.** Écris-les en commentaire avant de corriger.

In [ ]:
start("d5")

In [ ]:
def assign_clusters(X, C):
    d = ((X - C) ** 2).sum(-1)
    return d.argmin(dim=0)

def run_kmeans(X, K, n_iter=20):
    C = X[torch.randperm(len(X))[:K]]
    for _ in range(n_iter):
        a = assign_clusters(X, C)
        for k in range(K):
            C[k] = X[a == k].mean(0)
    return C, a

In [ ]:
check("d5")

## Bonus algo : BFS sur une grille  `bfs`

⏱ objectif 10 min

shortest_path(grid, start, goal) : plus court chemin en 4-connexité ('.' libre, '#' mur), −1 si impossible. Signalé une fois (stage, mars 2026) avec un K-Means ; Ilyas dit qu'il n'y a pas de LeetCode en applied science, donc 10 minutes suffisent.

In [ ]:
start("bfs")

In [ ]:
def shortest_path(grid, start, goal):
    """grid : liste de chaînes ('.' libre, '#' mur), déplacements en 4-connexité.
    Renvoie la longueur du plus court chemin de start à goal (tuples (ligne, colonne)), −1 si impossible."""
    raise NotImplementedError

In [ ]:
check("bfs")

## Distances au carré par paires  `t08`

⏱ objectif 4 min

(N, d) et (K, d) → (N, K), sans tenseur (N, K, d). Sans boucle Python.

In [ ]:
def pairwise_sq_dist(X, C):
    """X : (N, d), C : (K, d) → (N, K) distances au carré, sans créer de tenseur (N, K, d)."""
    raise NotImplementedError

In [ ]:
check("t08")

# Séance 10 · Mixture of Experts (Mixtral)

Écrire une couche MoE top-2 avec dispatch creux et loss d'équilibrage, comme Mixtral.

## Couche MoE top-2 (Mixtral)  `moe`

⏱ objectif 30 min

MoE(d_model, d_ff, n_experts, top_k) avec gate (nn.Linear sans biais) et experts (nn.ModuleList d'Expert, fourni). forward(x (B, T, D)) → (y, aux). Poids = softmax sur les k logits retenus (Mixtral). aux = E · Σ f_i · P_i. Chaque expert ne traite que ses tokens (dispatch creux, vérifié par le test).

In [ ]:
start("moe")

In [ ]:
class Expert(nn.Module):          # fourni : un FFN SwiGLU
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MoE(nn.Module):
    def __init__(self, d_model, d_ff, n_experts=8, top_k=2):
        super().__init__()
        # TODO : self.gate = nn.Linear(d_model, n_experts, bias=False) ; self.experts = nn.ModuleList(...)
        raise NotImplementedError

    def forward(self, x):
        """x : (B, T, D) → (y (B, T, D), aux_loss).
        Routage Mixtral : top-k des logits du routeur, poids = softmax sur ces k logits.
        aux = E · Σ_i f_i · P_i, f_i = part des affectations allant à i, P_i = proba moyenne du routeur.
        Chaque expert ne doit traiter que les tokens qui lui sont routés."""
        raise NotImplementedError

In [ ]:
check("moe")

## Refaire de zéro : KV cache et boucle de génération  `kv_cache`

Sans regarder ta version précédente : ajoute une cellule, recopie le squelette depuis la séance 7, `start("kv_cache")`, puis `check("kv_cache")`.

# Séance 11 · Fine-tuning et alignement

Écrire LoRA et les pertes DPO, GRPO et PPO clippé, et connaître la recette de Magistral.

## LoRA sur un nn.Linear  `lora`

⏱ objectif 15 min

LoRALinear(base, r, alpha) : base gelé, lora_A (r, in) en Kaiming, lora_B (out, r) à zéro, scaling = alpha / r. forward = base(x) + scaling · x Aᵀ Bᵀ. merge() renvoie un nn.Linear équivalent.

In [ ]:
start("lora")

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        # TODO : garder base (gelé), lora_A (r, in) init Kaiming, lora_B (out, r) à zéro, scaling = alpha / r
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

    def merge(self):
        """Renvoie un nn.Linear équivalent avec W + scaling · B A (pour l'inférence)."""
        raise NotImplementedError

In [ ]:
check("lora")

## Pertes RL : DPO, GRPO, PPO clippé  `rl`

⏱ objectif 20 min

sequence_logprobs (somme masquée des log p), dpo_loss, grpo_advantages (normalisation dans le groupe, std non biaisé), ppo_clip_loss avec clip-higher (ε_bas 0,2, ε_haut 0,28) normalisé par le nombre total de tokens, comme dans Magistral.

In [ ]:
start("rl")

In [ ]:
def sequence_logprobs(logits, labels, mask):
    """logits : (B, T, V) qui prédisent labels : (B, T) (déjà alignés) ; mask : (B, T), 1 sur les tokens
    de réponse → (B,) somme des log p(label)."""
    raise NotImplementedError

def dpo_loss(pol_chosen, pol_rejected, ref_chosen, ref_rejected, beta=0.1):
    """Entrées (B,) : log-probas de séquence. Renvoie la perte DPO moyenne."""
    raise NotImplementedError

def grpo_advantages(rewards, eps=1e-6):
    """rewards : (n_prompts, G) → avantages normalisés dans chaque groupe (écart-type de torch, non biaisé)."""
    raise NotImplementedError

def ppo_clip_loss(logp_new, logp_old, adv, mask, eps_low=0.2, eps_high=0.28):
    """Objectif PPO clippé par token (clip-higher asymétrique), normalisé par le nombre total de tokens."""
    raise NotImplementedError

In [ ]:
check("rl")

# Séance 12 · Tokenisation, lois d'échelle, évaluation

Écrire un BPE au niveau octet, raisonner avec Chinchilla et 6ND, et savoir évaluer un modèle.

## Tokenizer BPE niveau octet  `bpe`

⏱ objectif 30 min

train_bpe(text, vocab_size) sur les octets UTF-8 : fusionne la paire adjacente la plus fréquente (égalité : la première rencontrée), nouvel id 256 + i ; renvoie merges {(a, b): id}. encode applique la fusion de plus petit rang présente jusqu'à épuisement ; decode reconstruit le texte.

In [ ]:
start("bpe")

In [ ]:
def train_bpe(text, vocab_size):
    """BPE au niveau octet (UTF-8). Part des 256 octets, fusionne vocab_size − 256 fois la paire
    adjacente la plus fréquente (égalité : celle rencontrée en premier). Nouvel id = 256 + i.
    Renvoie merges : dict {(a, b): nouvel_id} dans l'ordre de création."""
    raise NotImplementedError

def encode(text, merges):
    """Octets UTF-8, puis applique en priorité la fusion de plus petit rang présente, jusqu'à n'en plus trouver."""
    raise NotImplementedError

def decode(ids, merges):
    raise NotImplementedError

In [ ]:
check("bpe")

## Refaire de zéro : Boucle d'entraînement complète  `train`

Sans regarder ta version précédente : ajoute une cellule, recopie le squelette depuis la séance 8, `start("train")`, puis `check("train")`.